In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm
import pandas as pd
import numpy as np
from pathlib import Path
import json
from PIL import Image

# Detecta se está no kaggle ou local
import os
IN_KAGGLE = os.path.exists('/kaggle')

# Define diretórios de dados e saída
if IN_KAGGLE:
    DATA_DIR = Path('/kaggle/input/skin-cancer-mnist-ham10000')
    PROCESSED_DIR = Path('/kaggle/working/processed')
    OUTPUT_DIR = Path('/kaggle/working/outputs')
else:
    DATA_DIR = Path('../data/raw')
    PROCESSED_DIR = Path('../data/processed')
    OUTPUT_DIR = Path('../models')

# Cria diretórios de saída, se não existirem
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

# Define dispositivo (GPU se disponível, caso contrário CPU)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Dispositivo: {DEVICE}")
print(f"Kaggle: {IN_KAGGLE}")

In [ ]:
# Define configuração do modelo e treinamento
CONFIG = {
    'model_name': 'efficientnet_b0',
    'num_classes': 7,
    'image_size': 224,
    'batch_size': 32,
    'epochs': 30,
    'seed': 42,
}

torch.manual_seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])

In [ ]:
class SkinLesionDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = Path(img_dir)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Busca a imagem nas subpastas se necessário
        img_path = self.img_dir / f"{row['image_id']}.jpg"
        if not img_path.exists():
            for part in ['HAM10000_images_part_1', 'HAM10000_images_part_2',
                        'ham10000_images_part_1', 'ham10000_images_part_2']:
                candidate = self.img_dir / part / f"{row['image_id']}.jpg"
                if candidate.exists():
                    img_path = candidate
                    break

        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)

        return img, row['label']

In [ ]:
# Define transformações de dados para treinamento e validação
train_transform = transforms.Compose([
    transforms.Resize((CONFIG['image_size'], CONFIG['image_size'])),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((CONFIG['image_size'], CONFIG['image_size'])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

In [ ]:
# Define diretório de imagens
if IN_KAGGLE:
    img_dir = DATA_DIR / 'images'
else:
    img_dir = DATA_DIR / 'images'

df_train = pd.read_csv(PROCESSED_DIR / 'train.csv')
df_val = pd.read_csv(PROCESSED_DIR / 'val.csv')
df_test = pd.read_csv(PROCESSED_DIR / 'test.csv')

train_dataset = SkinLesionDataset(df_train, img_dir, train_transform)
val_dataset = SkinLesionDataset(df_val, img_dir, val_transform)
test_dataset = SkinLesionDataset(df_test, img_dir, val_transform)

train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'],
                          shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'],
                        shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'],
                         shuffle=False, num_workers=2, pin_memory=True)

print(f"Treino:    {len(train_dataset)} imagens")
print(f"Validação: {len(val_dataset)} imagens")
print(f"Teste:     {len(test_dataset)} imagens")

In [ ]:
# Define o modelo usando a biblioteca timm
model = timm.create_model(
    CONFIG['model_name'],
    pretrained=True,
    num_classes=CONFIG['num_classes']
)

# Congela todo o backbone
for param in model.parameters():
    param.requires_grad = False

# Descongela só o classificador
for param in model.classifier.parameters():
    param.requires_grad = True

model = model.to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Fase 1 — Parâmetros treináveis: {trainable:,} / {total:,}")

In [ ]:
with open(PROCESSED_DIR / 'class_weights.json') as f:
    class_weights = json.load(f)

weights_tensor = torch.tensor(
    [class_weights[str(i)] for i in range(CONFIG['num_classes'])],
    dtype=torch.float32
).to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=weights_tensor)

# Fase 1: lr alto só no classificador
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3
)

In [ ]:
from sklearn.metrics import f1_score

def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, all_preds, all_labels = 0, [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        all_preds.extend(outputs.argmax(1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return total_loss / len(loader), f1

def val_epoch(model, loader, criterion):
    model.eval()
    total_loss, all_preds, all_labels = 0, [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            all_preds.extend(outputs.argmax(1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return total_loss / len(loader), f1

In [ ]:
# Define o número de épocas para cada fase
PHASE1_EPOCHS = 10
PHASE2_EPOCHS = 20

history = {'train_loss': [], 'val_loss': [], 'train_f1': [], 'val_f1': [], 'phase': []}
best_val_f1 = 0

for epoch in range(PHASE1_EPOCHS + PHASE2_EPOCHS):

    # Transição para fase 2
    if epoch == PHASE1_EPOCHS:
        print("\n--- Iniciando Fase 2: fine-tuning parcial ---")

        # Descongela os últimos 3 blocos do backbone
        for param in model.parameters():
            param.requires_grad = False
        for param in model.blocks[-3:].parameters():
            param.requires_grad = True
        for param in model.classifier.parameters():
            param.requires_grad = True

        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"Parâmetros treináveis na fase 2: {trainable:,}")

        # lr muito menor para não destruir features aprendidas
        optimizer = torch.optim.Adam(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=1e-5
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=PHASE2_EPOCHS
        )
    # Treinamento e validação
    train_loss, train_f1 = train_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_f1 = val_epoch(model, val_loader, criterion)
    # Atualiza o scheduler apenas na fase 2
    if epoch >= PHASE1_EPOCHS:
        scheduler.step()
    # Define a fase atual
    phase = 1 if epoch < PHASE1_EPOCHS else 2
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_f1'].append(train_f1)
    history['val_f1'].append(val_f1)
    history['phase'].append(phase)
    
    marker = ''
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), OUTPUT_DIR / 'best_model.pth')
        marker = ' ✓ (melhor)'

    print(f"[F{phase}] Epoch {epoch+1:02d} — train_loss={train_loss:.4f} val_loss={val_loss:.4f} val_f1={val_f1:.4f}{marker}")

print(f"\nMelhor val_f1: {best_val_f1:.4f}")

In [ ]:
# Salvar histórico
import json

with open(OUTPUT_DIR / 'history.json', 'w') as f:
    json.dump(history, f, indent=2)

print("Histórico salvo.")